# 🧪 Phase 2 — Adversarial Stress-Testing & Red-Teaming
## Breaking the 100% Score: Discovering the True Failure Boundaries of Track 1

### 🎯 Objective
In our clean benchmark evaluation, our fine-tuned model (`Qwen-1.5B-Legal-Graph-Adapter`) achieved **100.00% Macro F1** on Task A (Triplet Extraction).

**Is the model genuinely robust, or was the evaluation too easy?**
In this notebook, we red-team the fine-tuned adapter against **40 adversarial challenge cases** across 4 failure-testing dimensions:
1. **Suite 1: Inverted & Passive Syntax** (10 cases) — Can it handle passive voice and fronted clauses without inverting Plaintiff $\leftrightarrow$ Defendant?
2. **Suite 2: Heavy Distractors & Noise** (10 cases) — Does real-world noise (attorney names, dollar sums, dates, patent numbers) leak into the extracted triples?
3. **Suite 3: Negative / Dismissed Cases** (10 cases) — When a lawsuit is dismissed with prejudice, does the model **hallucinate** a non-existent remedy because training always had one?
4. **Suite 4: Commercial Partnerships & Alliances** (10 cases) — When two companies form a friendly partnership, does the model trigger a **false positive** lawsuit?

---
### 🛠️ Hardware & Setup
- **Target Runtime**: Google Colab T4 GPU or local GPU
- **Model**: Base `Qwen/Qwen2.5-1.5B-Instruct` (4-bit NF4) + `models/qwen-1.5b-legal-graph-adapter`
- **Execution Time**: ~5–8 minutes total

### Step 1: Install Dependencies & Hardware Audit

In [1]:
# Run in Google Colab:
!pip install -q transformers peft bitsandbytes accelerate tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.2 MB/s eta 0:00:00:00:0100:01


In [2]:
import sys
import os
import json
import re
import time
import torch
import transformers
from tqdm import tqdm

print("=" * 65)
print("🚀 ADVERSARIAL STRESS TEST — HARDWARE AUDIT")
print("=" * 65)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total           : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("GPU Device Name      : CPU Mode (Quantization disabled)")
print(f"Transformers Version : {transformers.__version__}")
print("=" * 65)

🚀 ADVERSARIAL STRESS TEST — HARDWARE AUDIT
Python Version       : 3.13.15
PyTorch Version      : 2.11.0+cu130
CUDA Available       : True
GPU Device Name      : Tesla T4
VRAM Total           : 14.56 GB
Transformers Version : 5.18.0


### Step 2: Locate Benchmark Files and Adapter Directory

In [3]:
# If running in Colab, ensure repository is present
if not os.path.exists("llm-graph-ontology"):
    !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git
    if os.path.exists("Fine-Tuning-PoCs/llm-graph-ontology"):
        %cd Fine-Tuning-PoCs

def find_path(rel_path):
    candidates = [
        rel_path,
        f"../{rel_path}",
        f"llm-graph-ontology/{rel_path}",
        f"Fine-Tuning-PoCs/llm-graph-ontology/{rel_path}",
        f"/content/Fine-Tuning-PoCs/llm-graph-ontology/{rel_path}",
        f"/content/llm-graph-ontology/{rel_path}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Cannot find path: {rel_path}")

stress_test_file = find_path("data/benchmark/legal_adversarial_stress_test.json")
adapter_dir = find_path("models/qwen-1.5b-legal-graph-adapter")

with open(stress_test_file, "r", encoding="utf-8") as f:
    stress_cases = json.load(f)

print(f"✅ Loaded {len(stress_cases)} adversarial test cases from: {stress_test_file}")
print(f"✅ Located trained adapter directory at: {adapter_dir}")

Cloning into 'Fine-Tuning-PoCs'...
remote: Enumerating objects: 557, done.
remote: Counting objects: 100% (213/213), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 557 (delta 100), reused 168 (delta 61), pack-reused 344 (from 1)
Receiving objects: 100% (557/557), 133.58 MiB | 19.59 MiB/s, done.
Resolving deltas: 100% (240/240), done.
Updating files: 100% (159/159), done.
/content/Fine-Tuning-PoCs
✅ Loaded 40 adversarial test cases from: llm-graph-ontology/data/benchmark/legal_adversarial_stress_test.json
✅ Located trained adapter directory at: llm-graph-ontology/models/qwen-1.5b-legal-graph-adapter


### Step 3: Load Base Model in 4-bit NF4 & Attach Fine-Tuned Adapter

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading base model ({MODEL_ID}) in 4-bit NF4...")

tokenizer = AutoTokenizer.from_pretrained(adapter_dir, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        device_map="auto"
    )

print(f"Attaching fine-tuned adapter from {adapter_dir}...")
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()
print("✅ Fine-Tuned Model successfully loaded and ready for Stress-Testing!")

Loading base model (Qwen/Qwen2.5-1.5B-Instruct) in 4-bit NF4...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Attaching fine-tuned adapter from llm-graph-ontology/models/qwen-1.5b-legal-graph-adapter...
✅ Fine-Tuned Model successfully loaded and ready for Stress-Testing!


### Step 4: Define Inference Helper & JSON Extraction Parser

In [5]:
def extract_triples_inference(text_passage):
    prompt_messages = [
        {
            "role": "system",
            "content": (
                "You are a formal corporate litigation knowledge graph extractor. "
                "Extract factual triples strictly conforming to the legal ontology relations: "
                "[FILES_CLAIM_AGAINST, ADJUDICATED_BY, APPLIES_PRECEDENT, ESTABLISHES_REMEDY, LIABLE_FOR]. "
                "Output ONLY a valid JSON list of objects with keys: 'subject', 'predicate', 'object'."
            )
        },
        {
            "role": "user",
            "content": f"Extract legal triples from this text:\n{text_passage}"
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def parse_triples_json(raw_text):
    cleaned = re.sub(r'```(?:json)?', '', raw_text).strip('` \n')
    try:
        data = json.loads(cleaned)
        if isinstance(data, list):
            return set(
                (item.get('subject', '').strip().lower(), item.get('predicate', '').strip().lower(), item.get('object', '').strip().lower())
                for item in data if isinstance(item, dict)
            )
    except Exception:
        pass
    return set()

print("Inference and parsing functions compiled.")

Inference and parsing functions compiled.


### Step 5: Execute Batch Inference Over All 40 Adversarial Cases

In [6]:
print("=" * 65)
print("🚀 EXECUTING ADVERSARIAL STRESS TEST (40 CASES)")
print("=" * 65)

results = []
start_time = time.time()

for item in tqdm(stress_cases, desc="Adversarial Cases"):
    raw_out = extract_triples_inference(item['input_text'])
    pred_set = parse_triples_json(raw_out)
    target_set = set(
        (t['subject'].strip().lower(), t['predicate'].strip().lower(), t['object'].strip().lower())
        for t in item['target_triples']
    )
    
    # Standard Precision / Recall / F1
    if len(target_set) == 0 and len(pred_set) == 0:
        prec, rec, f1 = 1.0, 1.0, 1.0
    elif len(pred_set) == 0 or len(target_set) == 0:
        prec = 1.0 if len(pred_set) == 0 and len(target_set) == 0 else 0.0
        rec = 1.0 if len(pred_set) == 0 and len(target_set) == 0 else 0.0
        f1 = 0.0
    else:
        tp = len(pred_set.intersection(target_set))
        prec = tp / len(pred_set) if len(pred_set) > 0 else 0.0
        rec = tp / len(target_set) if len(target_set) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0

    results.append({
        "id": item["id"],
        "stress_suite": item["stress_suite"],
        "input_text": item["input_text"],
        "raw_output": raw_out,
        "pred_set": pred_set,
        "target_set": target_set,
        "prec": prec,
        "rec": rec,
        "f1": f1,
        "is_litigation": item.get("is_litigation", True),
        "expected_remedy_present": item.get("expected_remedy_present", True)
    })

total_time = time.time() - start_time
print(f"\n✅ Completed evaluation in {total_time/60:.2f} minutes!")

🚀 EXECUTING ADVERSARIAL STRESS TEST (40 CASES)


Adversarial Cases: 100%|██████████| 40/40 [07:47<00:00, 11.68s/it]


✅ Completed evaluation in 7.78 minutes!


### Step 6: Detailed Analysis by Failure Dimension

In [7]:
# Filter by suite
s1_res = [r for r in results if r["stress_suite"] == "suite_1_inverted_syntax"]
s2_res = [r for r in results if r["stress_suite"] == "suite_2_heavy_distractors"]
s3_res = [r for r in results if r["stress_suite"] == "suite_3_dismissed_cases"]
s4_res = [r for r in results if r["stress_suite"] == "suite_4_commercial_partnerships"]

def avg(vals):
    return sum(vals) / len(vals) if len(vals) > 0 else 0.0

# 1. Suite 1: Inverted / Passive Syntax Analysis
s1_f1 = avg([r["f1"] for r in s1_res])
s1_inversion_count = 0
for r in s1_res:
    for p in r["pred_set"]:
        if p[1] == "files_claim_against":
            # Check if (object, files_claim_against, subject) was predicted
            for t in r["target_set"]:
                if t[1] == "files_claim_against" and p[0] == t[2] and p[2] == t[0]:
                    s1_inversion_count += 1

# 2. Suite 2: Distractor Contamination Analysis
s2_f1 = avg([r["f1"] for r in s2_res])
distractor_keywords = ["sarah", "jenkins", "david", "vance", "marcus", "green", "quinn", "kirkland", "arthur", "judge", "$", "million", "docket"]
s2_contaminated_count = 0
for r in s2_res:
    contaminated = False
    for p in r["pred_set"]:
        for kw in distractor_keywords:
            if kw in p[0] or kw in p[2]:
                contaminated = True
                break
    if contaminated:
        s2_contaminated_count += 1

# 3. Suite 3: Dismissed Cases - Remedy Hallucination Rate
s3_f1 = avg([r["f1"] for r in s3_res])
s3_hallucinated_remedy_count = 0
for r in s3_res:
    has_hallucinated_remedy = False
    for p in r["pred_set"]:
        if p[1] in ["liable_for", "establishes_remedy"]:
            has_hallucinated_remedy = True
    if has_hallucinated_remedy:
        s3_hallucinated_remedy_count += 1

# 4. Suite 4: Partnerships - False Positive Litigation Rate
s4_false_positive_count = 0
for r in s4_res:
    if len(r["pred_set"]) > 0:
        s4_false_positive_count += 1

# Overall Adversarial Macro F1
overall_f1 = avg([r["f1"] for r in results])

### Step 7: Print Official Adversarial Reality Check Table

In [8]:
print("\n" + "=" * 75)
print("📊 OFFICIAL ADVERSARIAL STRESS-TEST SCORECARD: TRACK 1 REALITY CHECK")
print("=" * 75)
print(f"Clean Laboratory Benchmark Macro F1 (From Notebook 02) : 100.00%")
print(f"Adversarial Stress-Test Macro F1 (Overall 40 Cases)    : {overall_f1 * 100:.2f}%")
print(f"Real-World Robustness Drop (Δ)                         : {((overall_f1 - 1.0) * 100):.2f}%")
print("-" * 75)
print("🔍 FAILURE BREAKDOWN BY CHALLENGE SUITE:")
print("-" * 75)
print(f"Suite 1: Inverted / Passive Syntax Macro F1           : {s1_f1 * 100:.2f}%")
print(f"         Directional Inversions (Plaintiff/Defendant) : {s1_inversion_count} / {len(s1_res)} cases ({s1_inversion_count/len(s1_res)*100:.1f}%)")
print("-" * 75)
print(f"Suite 2: Heavy Distractors & Noise Macro F1           : {s2_f1 * 100:.2f}%")
print(f"         Distractor Contamination (Lawyers/Amounts)   : {s2_contaminated_count} / {len(s2_res)} cases ({s2_contaminated_count/len(s2_res)*100:.1f}%)")
print("-" * 75)
print(f"Suite 3: Negative / Dismissed Cases Macro F1          : {s3_f1 * 100:.2f}%")
print(f"         Remedy Hallucination Rate                    : {s3_hallucinated_remedy_count} / {len(s3_res)} cases ({s3_hallucinated_remedy_count/len(s3_res)*100:.1f}%)")
print("-" * 75)
print(f"Suite 4: Commercial Partnerships (Non-Litigation)     :")
print(f"         False Positive Lawsuit Prediction Rate       : {s4_false_positive_count} / {len(s4_res)} cases ({s4_false_positive_count/len(s4_res)*100:.1f}%)")
print("=" * 75)

# Export metrics
stress_report = {
    "clean_benchmark_f1": 1.00,
    "adversarial_overall_f1": round(overall_f1, 4),
    "suite_1_inverted_syntax": {
        "f1": round(s1_f1, 4),
        "directional_inversions": s1_inversion_count,
        "total": len(s1_res)
    },
    "suite_2_heavy_distractors": {
        "f1": round(s2_f1, 4),
        "distractor_contaminations": s2_contaminated_count,
        "total": len(s2_res)
    },
    "suite_3_dismissed_cases": {
        "f1": round(s3_f1, 4),
        "remedy_hallucinations": s3_hallucinated_remedy_count,
        "total": len(s3_res)
    },
    "suite_4_commercial_partnerships": {
        "false_positive_lawsuits": s4_false_positive_count,
        "total": len(s4_res)
    }
}

with open("legal_adversarial_stress_test_results.json", "w", encoding="utf-8") as f:
    json.dump(stress_report, f, indent=2)
print("Saved stress test metrics to legal_adversarial_stress_test_results.json")


📊 OFFICIAL ADVERSARIAL STRESS-TEST SCORECARD: TRACK 1 REALITY CHECK
Clean Laboratory Benchmark Macro F1 (From Notebook 02) : 100.00%
Adversarial Stress-Test Macro F1 (Overall 40 Cases)    : 61.67%
Real-World Robustness Drop (Δ)                         : -38.33%
---------------------------------------------------------------------------
🔍 FAILURE BREAKDOWN BY CHALLENGE SUITE:
---------------------------------------------------------------------------
Suite 1: Inverted / Passive Syntax Macro F1           : 73.33%
         Directional Inversions (Plaintiff/Defendant) : 8 / 10 cases (80.0%)
---------------------------------------------------------------------------
Suite 2: Heavy Distractors & Noise Macro F1           : 93.33%
         Distractor Contamination (Lawyers/Amounts)   : 0 / 10 cases (0.0%)
---------------------------------------------------------------------------
Suite 3: Negative / Dismissed Cases Macro F1          : 80.00%
         Remedy Hallucination Rate                 